# Week 5 Exercise — Research / Learning RAG

**Course notes & research digest Q&A**

A RAG assistant over your knowledge base (course notes, lecture summaries, or research docs). Answer questions using only the retrieved context.

**LLM backends:** Ollama (local), OpenRouter, or Anthropic (Claude).

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

load_dotenv(override=True)

True

## Config

- **KNOWLEDGE_BASE**: folder of `.md` (or other) docs. Default points to week5's `knowledge-base`; replace with your own course/research folder.
- **LLM_PROVIDER**: `"ollama"` | `"openrouter"` | `"anthropic"`.
- Set `OPENROUTER_API_KEY` in `.env` for OpenRouter; `ANTHROPIC_API_KEY` for Claude.

In [2]:
# Run notebook from week5/community-contributions/profe-ssor so paths resolve correctly
NOTEBOOK_DIR = Path(".").resolve()

# Knowledge base: week5 default, or your own path (e.g. "research_notes", "course_notes")
KNOWLEDGE_BASE = NOTEBOOK_DIR / "../../knowledge-base"
DB_NAME = str(NOTEBOOK_DIR / "research_vector_db")

LLM_PROVIDER = "ollama"  # "ollama" | "openrouter" | "anthropic"
OLLAMA_MODEL = "llama3.2"
OPENROUTER_MODEL = "openai/gpt-4o-mini"
ANTHROPIC_MODEL = "claude-3-5-haiku-20241022"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
RETRIEVAL_K = 6

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

## Ingest: load docs, chunk, embed, store in Chroma

Uses HuggingFace embeddings (no API key needed). Run once to build the vector store.

In [4]:
def load_documents():
    docs = []
    base = Path(KNOWLEDGE_BASE)
    if not base.exists():
        raise FileNotFoundError(f"Knowledge base not found: {base}. Point KNOWLEDGE_BASE to a folder of .md files.")
    for folder in base.iterdir():
        if folder.is_dir():
            loader = DirectoryLoader(
                str(folder),
                glob="**/*.md",
                loader_cls=TextLoader,
                loader_kwargs={"encoding": "utf-8"},
            )
            for doc in loader.load():
                doc.metadata["doc_type"] = folder.name
                docs.append(doc)
    return docs

text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

documents = load_documents()
chunks = text_splitter.split_documents(documents)
print(f"Loaded {len(documents)} documents, {len(chunks)} chunks.")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
)
print(f"Vector store saved to {DB_NAME}")

Loaded 76 documents, 884 chunks.
Vector store saved to /home/professor/projects/llm_engineering/week5/community-contributions/profe-ssor/research_vector_db


## Connect to existing vector store and set up retriever + LLM

If you already ran ingest above, you can skip re-running it and just load the store and pick the LLM.

In [5]:
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

if LLM_PROVIDER == "ollama":
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
elif LLM_PROVIDER == "openrouter":
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise ValueError("Set OPENROUTER_API_KEY in .env for OpenRouter.")
    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
        model=OPENROUTER_MODEL,
        temperature=0,
    )
elif LLM_PROVIDER == "anthropic":
    llm = ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0)
else:
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")

print(f"Using LLM: {LLM_PROVIDER}")

Using LLM: ollama


In [6]:
SYSTEM_PROMPT_TEMPLATE = """
You are a research and learning assistant. Answer the user's question using ONLY the context below (course notes, summaries, or documents).
If the context does not contain enough information, say so. Do not invent facts.

Context:
{context}
"""

In [7]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ])
    return response.content

## Try a question

In [8]:
answer_question("What is RAG?", [])

'I don\'t have enough information to determine what "RAG" refers to in the context of the Contract with Harmony Health Plans for Healthllm. The provided text does not contain any definitions or explanations for this term. If you could provide more context or clarify what "RAG" stands for, I would be happy to try and assist you further.'

## Chat UI (Gradio)

In [9]:
gr.ChatInterface(answer_question, title="Research / Learning Q&A").launch()

/home/professor/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
